# Week 5 へようこそ - エージェントフレームワーク

## Day 2: AWS Strands

同じ週、同じ5つのステップ、新しいフレームワークです。今週の考え方全体は、ひとつのエージェントフレームワークを理解すれば他もだいたい理解できるということなので、毎日同じ5つのステップで同じエージェントを作り、その作法が響き合う様子を観察します。

1. **エージェントを作る** - モデルとシステムプロンプトを与える。
2. **実行する** - メッセージを送り、返信を受け取る。
3. **ツールを追加する** - エージェントが呼び出せる、普通の型付き関数。
4. **MCP を追加する** - 誰か他の人が書いたツールサーバーに接続する。毎回同じ方法でつなげる。
5. **ゴールを与えてループさせる** - 目標を渡し、仕事が終わるまで一歩ずつ自分で進めさせる。

ステップ1と2は、まだ単なる LLM 呼び出しです。ツールと MCP は、エージェントにできることを与えます。ステップ5でようやくエージェントらしくなります。フレームワーク自身がループを回し、ツールを選び、結果を読み、また選び直す、というのをゴールに到達するまで続けるのです。

実習プロジェクトは Day 1 と同じ SQLite の todo ボードです。ワーカーがボードから1つのゴールを取り出し、自分でステップを計画し、自分のエージェントループでその作業をこなし、各ステップにチェックを入れていきます。ボードのコード(`board.py`)は一字一句まったく同じファイルで、変わるのはそれを取り巻くフレームワークだけです。

今日は **AWS Strands** です。モデル駆動のループを中心に組まれた、軽量でオープンソースの SDK です。エージェントにタスクといくつかのツールを渡すと、モデル自身が計画を立て、ツールを呼び、結果を読み、完了するまで続けます。Strands は2つのことで知られています。この最小限のループと、本当の意味でのポータビリティです。ツールは一度書けば、裏側のモデルは1行で切り替えられます。この切り替えは今日の最後に見ることになります。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">AWS Strands のドキュメント</h2>
            <span style="color:#00bfff;">ドキュメントは <a href="https://strandsagents.com">https://strandsagents.com</a> にあり、整理されていて短くまとまっています。Strands はほぼ毎週新しいリリースを出すので、ここではバージョンを固定(1.43.0)し、古いブログ記事は注意して扱ってください。いくつかの API が最近変更されています。1つ注意点として、モデルを指定しない素の <code>Agent()</code> はデフォルトで AWS Bedrock を使おうとするので、ここでは常にモデルを明示的に渡します。</span>
        </td>
    </tr>
</table>

## セットアップ

今日必要なものは2つですが、どちらも以前の週からすでに用意されています。

- **Node**。`npx` のために必要です(filesystem MCP サーバーはこれを使って動きます)。`node --version` で確認してください。
- リポジトリのルートにある `.env` の中の **`OPENAI_API_KEY`**。今日のフレームワークは OpenAI の `gpt-5.4-mini` を使いますが、そのキーは Week 1 以来ずっと `.env` にあるはずです。

Strands はリポジトリの環境に含まれているので、リポジトリのルートで通常の `uv sync` を実行すればすべてインストールされます。このノートブックを Cursor で開き、毎週使っているリポジトリ既定の **Python 3.12.12** カーネルを選んで、上から順にセルを実行してください。

最初の実行を速くするために、今のうちに一度 filesystem MCP サーバーをウォームアップしておき、動作中と表示されたらすぐに Ctrl-C で止めてください。

```bash
npx -y @modelcontextprotocol/server-filesystem .
```

In [ ]:
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

load_dotenv(override=True)

## ステップ1: エージェントを作る

Strands では、エージェントは `Agent` です。モデルと `system_prompt` を持ちます。ここではモデルを `gpt-5.4-mini` を指す `OpenAIModel` として一度だけ組み立て、それをどこでも再利用します。最初に知っておくべきことがひとつあります。モデルを指定しない素の `Agent()` は、こっそり AWS Bedrock をデフォルトとして使おうとし、AWS の認証情報を期待します。そのため、ここでは常に `model=` を明示的に渡します。

In [ ]:
MODEL = "gpt-5.4-mini"

model = OpenAIModel(client_args={"api_key": os.environ["OPENAI_API_KEY"]}, model_id=MODEL)

agent = Agent(
    model=model,
    system_prompt="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## ステップ2: 実行する

メッセージを送り、返信を待ちます。Strands は生成されたテキストをその場でストリーミングするので、リアルタイムに現れる様子を観察できます。まだツールがないので、ループするものは何もなく、エージェントはただ答えるだけです。これはまだ単なる LLM 呼び出しです。素の `agent("...")` でも動きますが、ノートブックの中では Jupyter のイベントループと協調するように `invoke_async` を await します。

In [ ]:
result = await agent.invoke_async("Say hello in Spanish.")

## 今週のプロジェクト: SQLite の todo ボード

ワーカーは、Day 1 と同じ小さな SQLite ボード、同じ `board.py` ファイルを介して連携します。1つのファイル、1つのテーブルで、サーバーを立てる必要もありません。ワーカーには1つの**ゴール**が与えられ、それを達成するために自分自身の**ステップ**の todo をそのゴールの下に書き出し、進めるごとにチェックを入れていき、最後にゴールを完了にします。内部的にはボードは単なる辞書のリストです(ゴールの `parent_id` は None で、ステップは自分のゴールを指します)。

In [ ]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

`show_board()` は、Week 1 で使ったのと同じ rich スタイルで、その同じデータを綺麗に表示します。各ゴールの下にステップがインデントされて並び、完了したタスクは緑色の打ち消し線、進行中のタスクは黄色で表示されます。まだステップはありません。エージェントが計画を立てるときに自分でステップを書き出します。

In [ ]:
board.show_board()

## ステップ3: ツールを追加する

Strands でのツールは、`@tool` デコレーターでラップされた型付き Python 関数です。型ヒントは引数のスキーマになり、docstring はモデルが読む説明文になります(その `Args:` セクションが各パラメータを説明します)。そして、その関数をエージェントの `tools=[...]` リストに渡すだけです。

ここでは3つの小さなボードツールを書きます。ボードを読む `show_todos`、ゴールをステップに分解する `plan_steps`、todo を完了にする `complete_task` です。まずは簡単なエージェントに2つだけ与えて、ボードに何があるか尋ねてみましょう。答える前に自分から `show_todos` を呼び出すことを、自分の目で確かめてください。この「決める、呼ぶ、読む、答える」というサイクルこそ、エージェントループが回り始めた瞬間です。3つのツールすべてはステップ5で一緒になります。

In [ ]:
@tool
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board.

    Args:
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps to take, in order.
    """
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) done and record a short result summary.

    Args:
        task_id: The id of the todo to mark done.
        result: A short summary of what was accomplished.
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
board_agent = Agent(
    model=model,
    system_prompt="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [ ]:
result = await board_agent.invoke_async("What is on the board right now, and what is its status?")

## ステップ4: MCP を追加する

MCP は、単に「自分が書いていないツール」を、小さなプロトコル越しに接続したものです。今週すべてのフレームワークで使う同じ Node サーバーである filesystem リファレンスサーバーを、単一の `workspace` フォルダに限定してエージェントに与えます。これにより、エージェントはそのフォルダ内のファイルしか触れなくなります。Strands では、サーバーを `MCPClient` でラップして同じ `tools=[...]` リストに追加するだけで、接続のライフサイクルはStrandsが管理してくれます。

サーバーには `errlog=subprocess.DEVNULL` を渡しています。これによりサーバーの起動時ログが静かになるだけでなく、Windows 上の Jupyter カーネルからサーバーを実行できるようにもなります。Windows のカーネルの stderr には、実際のファイルディスクリプタがないためです。Mac と Linux では、これは何も変わりません。

In [ ]:
workspace = Path("workspace").resolve()   # エージェントが触れてよい唯一のフォルダ

filesystem = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
            cwd=str(workspace),  # 相対ファイル名がそこで解決されるよう、workspace内でサーバーを起動する
        ),
        errlog=subprocess.DEVNULL,
    ),
    startup_timeout=60
)

In [ ]:
file_agent = Agent(
    model=model,
    system_prompt="You can read and write files in your workspace. Use your tools to do what is asked.",
    tools=[filesystem],
)

In [ ]:
result = await file_agent.invoke_async("Read notes.txt and summarize it in one short sentence.")

## ステップ5: ゴールを与えてループさせる

さあ、いよいよ本番です。1つのエージェントに3つのボードツールすべてと filesystem サーバーを与え、ゴールを渡して、実行させましょう。エージェントは自分でボード上にステップを計画し、ファイルツールでそれを片付け、それぞれにチェックを入れ、作業が終わったらゴールを完了にします。これこそ、自律的に動くエージェントループです。読む、計画する、行動する、チェックする、繰り返す。ボードにステップが埋まり、それが打ち消し線で消されていく様子を観察してください。

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

await worker.invoke_async("Please work the pending goal on the board.")
board.show_board()

## 同じワーカーをターミナルから実行する

ステップ5でたった今観察した内容はすべて、このノートブックの隣にある小さなスクリプト `strands_worker.py` としてもパッケージ化されています。これは同じゴールを登録し、同じ3つのボードツールと filesystem MCP サーバーを使って同じエージェントを組み立て、カーネルではなくコマンドラインから同じループを実行します。このフォルダでターミナルを開いて実行してください。

```bash
uv run strands_worker.py
```

エージェントがステップを計画し、ゴールに取り組んでそれぞれにチェックを入れていく様子、そして完成したボードと、書き出されたスペイン語が表示されます。これは Day 5 でどのワーカーも取る形と同じです。Day 5 では、Google ADK のオーケストレーターが、フレームワークごとにこうしたワーカーを1つずつ、共有された1つのボードに対する並列サブプロセスとして起動します。

## いちばん興味深い点: モデルを1行で切り替える

Strands の看板機能はポータビリティです。ツールとループは一度書けば終わりです。まったく同じエージェントを別のプロバイダーで動かすには、1行、`OpenAIModel` を変更するだけです。`base_url` を任意の OpenAI 互換エンドポイントに向ければ、他は何も変わりません。

```python
# 同じエージェントを OpenRouter、Ollama、LM Studio、ローカルの vLLM サーバー、その他 OpenAI 互換のものすべてで動かす
model = OpenAIModel(
    client_args={
        "api_key": OPENROUTER_KEY,             # ローカルサーバーの場合は空でない任意の文字列でよい
        "base_url": "https://openrouter.ai/api/v1",
    },
    model_id="gpt-5.4-mini",                    # あるいはそのエンドポイントが提供するものなら何でも
)
```

Strands は Anthropic、Gemini、Bedrock にも、それぞれ専用のモデルクラスを通じて対応していますが、OpenAI 互換の `base_url` はエコシステムの大部分をカバーしています。ツール、filesystem MCP サーバー、そしてボードは何ひとつ変わりません。これこそ、たった1つのフレームワークの中に詰め込まれた今週全体のテーマです。エージェントはエージェントであり、その裏にあるモデルは差し替え可能な細部にすぎないのです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">エクササイズ</h2>
            <span style="color:#ff7800;">ボードに別のゴールを、たとえば「マドリードについての短い俳句を書いて madrid.txt に保存する」を登録し、ワーカーを再度実行してみましょう。ワーカーは適切なステップを計画し、正しいファイルツールを選べるでしょうか。次に、1行でのモデル切り替えを試してみましょう。<code>OpenAIModel</code> を、自分がアクセスできる別の OpenAI 互換エンドポイントに向け、ワーカーを再実行し、同じエージェントが別のモデルで動く様子を観察してみましょう。</span>
        </td>
    </tr>
</table>